# Notebook 07 — Common Biochemistry Assay Data Shapes

**Module:** 07 — Biochemistry and Molecular Biology  
**Notebook:** 07 of 10  
**Prerequisites:** NB02–NB06  
**Time estimate:** 75 minutes

> **Data literacy goal:** This notebook is about reading, not generating. By the end,
> you should be able to look at any common biochemistry assay figure and immediately
> identify what was measured, what the axes represent, and how to extract a number.

---
## Step 1 — Motivation

Computational biologists work with data generated by wet-lab assays. Every RNA-seq
expression value was validated by qRT-PCR. Every protein interaction was confirmed
by co-IP (co-immunoprecipitation) or a Western blot. Every drug target was characterised
by an enzyme assay that produced a dose-response curve. Being unable to interpret
these data types creates a communication gap with experimental collaborators.

---
## Step 3 — Biological Background

### Assay types and data shapes

**1. Western blot (WB / immunoblot)**
- Measures: presence and approximate amount of a specific protein by MW
- Data: gel image with bands; x-axis = sample lane, y-axis = MW (kDa)
- Readout: band intensity ∝ protein amount; band position ↔ MW
- Common artefact: non-specific bands, loading differences (need a loading control: GAPDH)

**2. ELISA (Enzyme-Linked ImmunoSorbent Assay)**
- Measures: concentration of an antigen (protein, cytokine, antibody) in a sample
- Data: absorbance vs. analyte concentration; sigmoidal dose-response curve
- Model: 4-parameter logistic (4PL): y = Bottom + (Top - Bottom) / (1 + (EC50/x)^n)
- Readout: EC50 = half-maximal effective concentration

**3. Gel electrophoresis (SDS-PAGE)**
- Measures: protein molecular weights
- Data: band migration distance is log-linear with MW
- Model: log(MW) = a − b × migration (linear fit to ladder standards)

**4. Flow cytometry**
- Measures: per-cell fluorescence intensity across multiple channels simultaneously
- Data: 2D scatter plot (FSC vs. SSC for scatter; channel vs. channel for markers)
- Readout: gates define cell populations; %positive, mean fluorescence intensity (MFI)

**5. LC-MS/MS (proteomics)**
- Measures: peptide masses and fragmentation patterns
- Data: (retention time, m/z, intensity) triplets; spectrum = m/z vs. intensity
- Readout: peptide identification via database search; protein quantification via
  peak area (label-free, LFQ) or reporter ion intensity (TMT/iTRAQ)

---
## Step 4 — Mathematical Explanation

**4PL model (ELISA):**
$$y = Bottom + \frac{Top - Bottom}{1 + \left(\frac{EC_{50}}{x}\right)^n}$$
- Bottom: minimum response (background)
- Top: maximum response (saturation)
- EC50: half-maximal concentration
- n: Hill slope (steepness of the transition)

**Gel migration model:**
$$\log_{10}(MW) = a - b \times d_{migration}$$
Fit a and b from the protein ladder, then infer MW from migration distance.

---
## Step 6 — Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.optimize import curve_fit

In [ ]:
# Cell 6.1 — ELISA: 4-parameter logistic model + fitting
def four_pl(x: np.ndarray, bottom: float, top: float,
            ec50: float, n: float) -> np.ndarray:
    """4-Parameter Logistic (4PL) model for ELISA dose-response."""
    return bottom + (top - bottom) / (1 + (ec50 / x)**n)

# Simulate ELISA standard curve data
rng = np.random.default_rng(42)
conc_std = np.array([0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0, 300.0])  # pg/mL
true_params = (0.05, 2.5, 10.0, 1.3)  # bottom, top, EC50, Hill
absorbance_true = four_pl(conc_std, *true_params)
absorbance_obs = absorbance_true + rng.normal(0, 0.05, len(conc_std))
absorbance_obs = np.clip(absorbance_obs, 0.01, 3.0)

# Fit 4PL to data
popt, pcov = curve_fit(four_pl, conc_std, absorbance_obs,
                        p0=[0.05, 2.5, 10, 1.2],
                        bounds=([0, 0, 0, 0], [1, 5, 1000, 5]))
bottom_fit, top_fit, ec50_fit, n_fit = popt
print(f"4PL fit:")
print(f"  EC50 = {ec50_fit:.2f} pg/mL  (true: {true_params[2]})")
print(f"  Hill slope n = {n_fit:.2f}  (true: {true_params[3]})")

# Interpolate unknown sample concentrations
sample_absorbances = [0.38, 0.92, 1.77]
def inverse_4pl(y, bottom, top, ec50, n):
    """Back-calculate concentration from absorbance using 4PL."""
    return ec50 * ((top - bottom) / (y - bottom) - 1) ** (-1/n)

print("\nUnknown sample interpolation:")
for abs_val in sample_absorbances:
    conc = inverse_4pl(abs_val, *popt)
    print(f"  A={abs_val:.2f} → concentration = {conc:.2f} pg/mL")

In [ ]:
# Cell 6.2 — Gel electrophoresis: MW from migration distance
def gel_mw_from_migration(migration_distance: np.ndarray,
                            ladder_mw: np.ndarray,
                            ladder_migration: np.ndarray) -> np.ndarray:
    """
    Estimate protein MW from SDS-PAGE migration distance.
    Uses log-linear fit to ladder standards.
    """
    log_mw = np.log10(ladder_mw)
    # Fit: log(MW) = a + b * migration
    coeffs = np.polyfit(ladder_migration, log_mw, 1)
    log_mw_unknown = np.polyval(coeffs, migration_distance)
    return 10**log_mw_unknown

# Standard protein ladder
ladder_mw_kda  = np.array([10, 15, 20, 25, 37, 50, 75, 100, 150, 250])
ladder_migr_cm = np.array([7.2, 6.5, 5.9, 5.4, 4.6, 3.9, 3.1, 2.5, 1.9, 1.2])

# Unknown samples
unknown_migrations = np.array([4.3, 3.5, 2.2])
estimated_mws = gel_mw_from_migration(unknown_migrations, ladder_mw_kda, ladder_migr_cm)

print("SDS-PAGE MW estimation from migration:")
for migr, mw in zip(unknown_migrations, estimated_mws):
    print(f"  Migration = {migr} cm → MW ≈ {mw:.1f} kDa")

In [ ]:
# Cell 6.3 — Simulate LC-MS/MS spectrum (peptide fragmentation)
def simulate_ms2_spectrum(peptide_seq: str, charge: int = 2,
                           seed: int = 42) -> dict:
    """
    Simulate a simplified MS2 spectrum for a peptide.
    Generates b-ions (N-terminal) and y-ions (C-terminal).

    Real mass spec: peptide fragmented at amide bonds → b/y ion series.
    b_n = sum(residue masses, 1..n) + 1.0079  (proton)
    y_n = sum(residue masses from C-term, 1..n) + 18.011 + 1.0079
    """
    # Simplified residue masses (monoisotopic)
    RESIDUE_MASSES = {
        'A':71.037, 'R':156.101, 'N':114.043, 'D':115.027, 'C':103.009,
        'Q':128.058, 'E':129.043, 'G':57.021, 'H':137.059, 'I':113.084,
        'L':113.084, 'K':128.095, 'M':131.040, 'F':147.068, 'P':97.053,
        'S':87.032, 'T':101.048, 'W':186.079, 'Y':163.063, 'V':99.068,
    }
    H = 1.00782
    H2O = 18.01056

    seq = peptide_seq.upper()
    masses = [RESIDUE_MASSES.get(aa, 0) for aa in seq]

    b_ions = {}
    for i in range(1, len(seq)):
        b_mass = sum(masses[:i]) + H
        b_ions[f'b{i}'] = b_mass / charge

    y_ions = {}
    for i in range(1, len(seq)):
        y_mass = sum(masses[len(seq)-i:]) + H2O + H
        y_ions[f'y{i}'] = y_mass / charge

    rng = np.random.default_rng(seed)
    # Assign random intensities (some ions are more abundant)
    all_ions = {**b_ions, **y_ions}
    intensities = {k: rng.exponential(1.0) * (5 if k.startswith('y') else 3)
                   for k in all_ions}

    return {'ions': all_ions, 'intensities': intensities,
            'peptide': seq, 'charge': charge}

spec = simulate_ms2_spectrum('ACDEFGHIK', charge=2)
print(f"Peptide: {spec['peptide']}  (charge +{spec['charge']})")
print(f"Number of theoretical ions: {len(spec['ions'])}")
top5 = sorted(spec['intensities'], key=lambda k: -spec['intensities'][k])[:5]
print("Top 5 ions by intensity:")
for ion in top5:
    print(f"  {ion}: m/z = {spec['ions'][ion]:.3f}, intensity = {spec['intensities'][ion]:.3f}")

---
## Step 7 — Visualization

In [ ]:
# Cell 7.1 — 4-panel: ELISA + Gel + Flow cytometry + MS2 spectrum
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Panel A: ELISA dose-response
ax = axes[0, 0]
x_fit = np.logspace(-1, 2.6, 200)
y_fit = four_pl(x_fit, *popt)
ax.semilogx(x_fit, y_fit, color='steelblue', lw=2, label='4PL fit')
ax.scatter(conc_std, absorbance_obs, color='tomato', s=50, zorder=5, label='Standard data')
ax.axhline(popt[0] + (popt[1]-popt[0])/2, color='gray', ls=':', lw=1)
ax.axvline(ec50_fit, color='gray', ls=':', lw=1)
ax.text(ec50_fit * 1.3, 0.3, f'EC50={ec50_fit:.1f} pg/mL', fontsize=8)
ax.set_xlabel('[Analyte] (pg/mL)'); ax.set_ylabel('Absorbance (450 nm)')
ax.set_title('A. ELISA standard curve (4PL)')
ax.legend(fontsize=8)

# Panel B: SDS-PAGE gel simulation
ax = axes[0, 1]
ax.set_xlim(-0.5, 4); ax.set_ylim(0, 8)
ax.set_facecolor('#f5f0e8')  # gel background
ax.set_xticks([])
ax.set_title('B. SDS-PAGE: MW vs. migration')

# Ladder bands
for mw, migr in zip(ladder_mw_kda, ladder_migr_cm):
    ax.barh(migr, 0.8, left=0.1, height=0.1, color='gray', alpha=0.7)
    ax.text(0.95, migr, f'{mw} kDa', va='center', fontsize=7)

# Sample bands
sample_colors = ['steelblue', 'tomato', 'seagreen']
for i, (migr, mw, color) in enumerate(zip(unknown_migrations, estimated_mws, sample_colors)):
    ax.barh(migr, 0.8, left=1.2 + i*0.9, height=0.15, color=color, alpha=0.8)
    ax.text(1.6 + i*0.9, migr + 0.2, f'~{mw:.0f} kDa', va='bottom', ha='center', fontsize=7)

ax.set_ylabel('Migration distance from well (cm)')
ax.text(0.5, 0.1, 'Ladder', ha='center', fontsize=8, color='gray')
for i, label in enumerate(['S1', 'S2', 'S3']):
    ax.text(1.6 + i*0.9, 0.1, label, ha='center', fontsize=8)
ax.invert_yaxis()

# Panel C: Flow cytometry scatter
ax = axes[1, 0]
rng2 = np.random.default_rng(7)
n_cells = 3000

# Population 1: lymphocytes (low FSC, low SSC)
fsc1 = rng2.normal(200, 30, n_cells//3)
ssc1 = rng2.normal(100, 20, n_cells//3)

# Population 2: monocytes (high FSC, medium SSC)
fsc2 = rng2.normal(400, 40, n_cells//3)
ssc2 = rng2.normal(250, 40, n_cells//3)

# Population 3: granulocytes (high FSC, high SSC)
fsc3 = rng2.normal(450, 50, n_cells//3)
ssc3 = rng2.normal(500, 60, n_cells//3)

ax.scatter(fsc1, ssc1, s=2, alpha=0.4, color='steelblue', label='Lymphocytes')
ax.scatter(fsc2, ssc2, s=2, alpha=0.4, color='tomato', label='Monocytes')
ax.scatter(fsc3, ssc3, s=2, alpha=0.4, color='seagreen', label='Granulocytes')
ax.set_xlabel('Forward Scatter (FSC) — cell size')
ax.set_ylabel('Side Scatter (SSC) — granularity')
ax.set_title('C. Flow cytometry: PBMC scatter')
ax.legend(fontsize=7, markerscale=4)

# Panel D: MS2 spectrum (annotated)
ax = axes[1, 1]
ions = spec['ions']; intensities = spec['intensities']
mz_vals = list(ions.values())
int_vals = [intensities[k] for k in ions.keys()]
labels = list(ions.keys())

for mz, inten, label in zip(mz_vals, int_vals, labels):
    color = 'steelblue' if label.startswith('b') else 'tomato'
    ax.vlines(mz, 0, inten, colors=color, lw=1.5)
    if inten > 3.5:
        ax.text(mz, inten + 0.1, label, ha='center', fontsize=6, color=color)

ax.set_xlabel('m/z')
ax.set_ylabel('Intensity')
ax.set_title(f'D. MS2 spectrum: {spec["peptide"]} (+{spec["charge"]})')
b_patch = mpatches.Patch(color='steelblue', label='b-ions (N-terminal)')
y_patch = mpatches.Patch(color='tomato', label='y-ions (C-terminal)')
ax.legend(handles=[b_patch, y_patch], fontsize=8)

plt.suptitle('Common Biochemistry Assay Data Shapes', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

---
## Step 8 — Exercises

1. Implement `four_pl(x, bottom, top, ec50, n)` and `inverse_4pl()`. Given an ELISA
   standard curve with EC50 = 50 pg/mL, what is the expected absorbance at 200 pg/mL?
2. Implement `gel_mw_from_migration()`. A band migrates 3.8 cm in a gel where the
   ladder has 50 kDa at 3.9 cm and 75 kDa at 3.1 cm. Estimate the unknown MW.
3. A Western blot shows a band at 120 kDa for an antibody targeting a protein
   predicted to be 45 kDa. List three biological and two technical explanations.
4. In flow cytometry, CD4⁺ T cells are defined as FSCᵐⁱᵈ SSCˡᵒ CD3⁺ CD4⁺.
   What is 'gating' and why is the order of gates important?

---
## Quiz — Active Recall

1. What does an ELISA measure? What is EC50?
2. In SDS-PAGE, why does migration distance correlate with log(MW)?
3. What are b-ions and y-ions in mass spectrometry? Which end of the peptide do each come from?
4. What is the loading control in a Western blot and why is it needed?
5. Flow cytometry can measure 20+ markers simultaneously. What technique enables this?

---
## Reflection

**Date completed:** ____________________

> *[A paper shows a Western blot of 'clean' data with no loading control and perfectly
> uniform bands. What concern does this raise, and what would you require as a reviewer?]*

---
*Next: `08_mini_project_michaelis_menten_simulation.ipynb`*